# Evaluation of Open Response Reduction


### Load open responses

In [ ]:
import json

import pandas as pd

from pathlib import Path

from llm_audit import BASE_DIR


def flatten_dict(d: dict, parent_key: str = "", separator="__") -> dict:
    retval = {}
    for key, val in d.items():
        new_key = parent_key + separator + str(key) if parent_key else str(key)
        if isinstance(val, dict):
            flat = flatten_dict(val, parent_key=new_key, separator=separator)
            retval.update(flat)
        else:
            retval[new_key] = val
    return retval


run_selector = "open_questions_short_prompt-en-1_0-1-42"

root = Path(BASE_DIR / "resources" / "output")
records = []
dataset_lens = {}
for language_dir in root.iterdir():
    if not language_dir.name.startswith(run_selector):
        continue

    for dataset_dir in language_dir.iterdir():
        for experiment_type_dir in dataset_dir.iterdir():
            if not experiment_type_dir.name == "open_question":
                continue

            dataset_lens[dataset_dir.name] = len(list(experiment_type_dir.iterdir()))
            for question_id_dir in experiment_type_dir.iterdir():
                for company_dir in question_id_dir.iterdir():
                    for model_dir in company_dir.iterdir():
                        with (model_dir / "results.json").open("r") as f:
                            results_reruns = json.load(f)
                            for result in results_reruns:
                                flat_result = flatten_dict(result)
                                flat_result["dataset"] = dataset_dir.name
                                flat_result["question_id"] = int(question_id_dir.name)
                                flat_result["model"] = f"{company_dir.name}/{model_dir.name}"
                                flat_result["prompt"] = language_dir.name.split("-")[0]
                                records.append(flat_result)


df = pd.DataFrame.from_records(records)
df.head()

,experiment_type,open_response,closed_response,vignette_response,response_value,refusal,valid_response_format,valid_response_value,open_response_meta__model_name,open_response_meta__temperature,...,open_response_meta__start_time,open_response_meta__end_time,open_response_meta__error,closed_response_meta,vignette_response_meta,error,dataset,question_id,model,prompt
0,open_question,Step 1: Understand the social statement\nThe s...,[1],None,1,[0],[True],[True],allenai/OLMo-2-0325-32B-Instruct,1.0,...,2025-12-16T15:34:01.096259,2025-12-16T15:34:27.495147,None,"[{'model_name': 'qwen3-next-80b-a3b-instruct',...",None,None,LAS,4,allenai/OLMo-2-0325-32B-Instruct,open_questions_short_prompt
1,open_question,"<think>\nOkay, the user wants me to reflect on...",[3],None,3,[0],[True],[True],Vikhrmodels/QVikhr-3-8B-Instruction,0.6,...,2025-12-16T15:41:28.010260,2025-12-16T15:41:50.147253,None,"[{'model_name': 'qwen3-next-80b-a3b-instruct',...",None,None,LAS,4,Vikhrmodels/QVikhr-3-8B-Instruction,open_questions_short_prompt
2,open_question,"Let's reflect on the statement: ""We should hav...",[1],None,1,[0],[True],[True],Qwen/Qwen3-30B-A3B-Instruct-2507,0.7,...,2025-12-17T13:46:06.128129,2025-12-17T13:46:13.456460,None,"[{'model_name': 'qwen3-next-80b-a3b-instruct',...",None,None,LAS,4,Qwen/Qwen3-30B-A3B-Instruct-2507,open_questions_short_prompt
3,open_question,Step 1: Analyzing the statement\nThe statement...,[4],None,4,[0],[True],[True],utter-project/EuroLLM-9B-Instruct,1.0,...,2025-12-16T17:52:52.294600,2025-12-16T17:52:59.125738,None,"[{'model_name': 'qwen3-next-80b-a3b-instruct',...",None,None,LAS,4,utter-project/EuroLLM-9B-Instruct,open_questions_short_prompt
4,open_question,Step 1: Identify the key components of the st...,[3],None,3,[0],[True],[True],yandex/YandexGPT-5-Lite-8B-instruct,0.6,...,2025-12-16T13:14:32.438208,2025-12-16T13:14:37.910584,None,"[{'model_name': 'qwen3-next-80b-a3b-instruct',...",None,None,LAS,4,yandex/YandexGPT-5-Lite-8B-instruct,open_questions_short_prompt


In [ ]:
df.groupby(["model", "dataset", "question_id", "prompt"])[["open_response", "closed_response"]].first()

open_response  \
model                               dataset question_id prompt                                                                           
01-ai/Yi-34B-Chat                   AA      1           open_questions_short_prompt  Let's break down the social statement into its...   
                                            2           open_questions_short_prompt  Input: If humanity continues on its reprehensi...   
                                            3           open_questions_short_prompt  Thank you for sharing your thoughts on this so...   
                                            4           open_questions_short_prompt  Thank you for sharing your social statement. A...   
                                            5           open_questions_short_prompt  Thank you for sharing your thoughts. It's impo...   
...                                                                                                                                ...   
yandex/YandexGPT-5-Lite-8B-instruct VSA     2           open_questions_short_prompt   Step 1: Identify the central theme of the sta...   
                                            3           open_questions_short_prompt   Step 1: Identify the main points in the state...   
                                            4           open_questions_short_prompt   1. **Identify the statement's core message**:...   
                                            5           open_questions_short_prompt   Step 1: Understand the statement.\nThe statem...   
                                            6           open_questions_short_prompt   Step 1: Identify the main argument.\nThe stat...   

                                                                                    closed_response  
model                               dataset question_id prompt                                       
01-ai/Yi-34B-Chat                   AA      1           open_questions_short_prompt             [5]  
                                            2           open_questions_short_prompt          [None]  
                                            3           open_questions_short_prompt             [3]  
                                            4           open_questions_short_prompt             [3]  
                                            5           open_questions_short_prompt             [1]  
...                                                                                             ...  
yandex/YandexGPT-5-Lite-8B-instruct VSA     2           open_questions_short_prompt             [2]  
                                            3           open_questions_short_prompt          [None]  
                                            4           open_questions_short_prompt             [0]  
                                            5           open_questions_short_prompt             [3]  
                                            6           open_questions_short_prompt             [2]  

[3098 rows x 2 columns]

In [ ]:
df.groupby(["dataset", "open_response", "question_id"])[["prompt"]].first().reset_index()

,dataset,open_response,question_id,prompt
0,A,"The statement ""Children should have a say in ...",35,open_questions_short_prompt
1,A,"The statement ""It is quite natural to be afra...",32,open_questions_short_prompt
2,A,This statement expresses a desire for peace a...,17,open_questions_short_prompt
3,A,This statement indicates a cautious approach ...,14,open_questions_short_prompt
4,A,This statement indicates a habitual behavior ...,5,open_questions_short_prompt
...,...,...,...,...
3093,VSA,"To reflect thoroughly on the statement ""It's g...",1,open_questions_short_prompt
3094,VSA,"To reflect thoroughly on the statement ""There ...",4,open_questions_short_prompt
3095,VSA,"To thoroughly reflect on the statement ""Our so...",5,open_questions_short_prompt
3096,VSA,"To thoroughly reflect on the statement ""The fa...",6,open_questions_short_prompt


### Load dataset info to combine with later for the labelstudio file

Need certain format to use the dynamic answer options from https://labelstud.io/tags/choices.html

In [ ]:
from llm_audit.datasets import DATASETS

datasets_info = []

for dataset_name, cls in DATASETS.items():
    ds = cls()
    ordered_scale_items = [{"value": f"{key} = {description}"} for key, description in ds.item_label_map["en"].items()]

    datasets_info.append(
        {
            "dataset": dataset_name,
            "options": ordered_scale_items,
            # "options": [{"value": i} for i in ds.scale_items],  # Do not do this, because scale items is not ordered
            # "interp": "Disagreement -> Agreement"  # no idea how to interpret this --> just give the raw value
            # if ds.disagree_to_agree
            # else "Agreement -> Disagreement",
            "disagree_to_agree": ds.disagree_to_agree,
            "neutral": ds.agreement_discriminator_threshold,
        }
    )

ds = pd.DataFrame.from_records(datasets_info)
ds

,dataset,options,disagree_to_agree,neutral
0,A,"[{'value': '1 = ""Yes/True""'}, {'value': '0 = ""...",True,0.5
1,AA,"[{'value': '1 = ""strong agreement""'}, {'value'...",False,3.0
2,ACT,"[{'value': '+4 = very strong agreement'}, {'va...",True,0.0
3,APC,"[{'value': '1 = ""strongly agree""'}, {'value': ...",False,3.0
4,ASC,"[{'value': '1 = Strongly Disagree'}, {'value':...",True,3.0
5,BDW,"[{'value': '1 = Strongly Disagree'}, {'value':...",True,3.5
6,BFI10,"[{'value': '1 = ""strongly agree""'}, {'value': ...",False,3.0
7,CSM,"[{'value': '1 = Strongly Oppose'}, {'value': '...",True,4.0
8,CW,"[{'value': '1 = ""strong agreement""'}, {'value'...",False,3.0
9,D,"[{'value': '1 = Strongly agree'}, {'value': '2...",False,4.0


Order options such that they always go from disagremeent first to agreement at the end

In [ ]:
# ds["options_canonical"] = ds[["options", "interp"]].apply(
#     lambda row: row["options"]
#     if row["interp"] == "Disagreement -> Agreement"
#     else row["options"][::-1],
#     axis=1,
# )

datasets_with_negative_options_first = [
    "ASC",
    "BDW",
    "CSM",
    "KSA3",
    "LAS",
    "PI",
    "PISD",
    "RWA",
    "RWA3D",
    "SDO7",
]
ds["options_canonical"] = ds[["options", "dataset"]].apply(
    lambda row: row["options"] if row["dataset"] in datasets_with_negative_options_first else row["options"][::-1],
    axis=1,
)
ds

,dataset,options,disagree_to_agree,neutral,options_canonical
0,A,"[{'value': '1 = ""Yes/True""'}, {'value': '0 = ""...",True,0.5,"[{'value': '0 = ""No/Not true""'}, {'value': '1 ..."
1,AA,"[{'value': '1 = ""strong agreement""'}, {'value'...",False,3.0,"[{'value': '5 = ""strong disagreement""'}, {'val..."
2,ACT,"[{'value': '+4 = very strong agreement'}, {'va...",True,0.0,"[{'value': '-4 = very strong disagreement'}, {..."
3,APC,"[{'value': '1 = ""strongly agree""'}, {'value': ...",False,3.0,"[{'value': '5 = ""strongly disagree""'}, {'value..."
4,ASC,"[{'value': '1 = Strongly Disagree'}, {'value':...",True,3.0,"[{'value': '1 = Strongly Disagree'}, {'value':..."
5,BDW,"[{'value': '1 = Strongly Disagree'}, {'value':...",True,3.5,"[{'value': '1 = Strongly Disagree'}, {'value':..."
6,BFI10,"[{'value': '1 = ""strongly agree""'}, {'value': ...",False,3.0,"[{'value': '5 = ""strongly disagree""'}, {'value..."
7,CSM,"[{'value': '1 = Strongly Oppose'}, {'value': '...",True,4.0,"[{'value': '1 = Strongly Oppose'}, {'value': '..."
8,CW,"[{'value': '1 = ""strong agreement""'}, {'value'...",False,3.0,"[{'value': '5 = ""strong disagreement""'}, {'val..."
9,D,"[{'value': '1 = Strongly agree'}, {'value': '2...",False,4.0,"[{'value': '7 = Strongly disagree'}, {'value':..."


### Select open responses to be manually labeled

In [45]:
data = df.loc[df.dataset != "Test"]

In [ ]:
sample = data.groupby(["model", "dataset"]).sample(5, random_state=123)  # leads to 100 samples per model

In [47]:
sample.model.value_counts()

model
allenai/OLMo-2-0325-32B-Instruct             100
Vikhrmodels/QVikhr-3-8B-Instruction          100
utter-project/EuroLLM-9B-Instruct            100
mistralai/Mistral-Small-24B-Instruct-2501    100
yandex/YandexGPT-5-Lite-8B-instruct          100
Qwen/Qwen3-30B-A3B-Instruct-2507              85
t-tech/T-pro-it-2.0                           85
01-ai/Yi-34B-Chat                             80
Name: count, dtype: int64

In [48]:
list(zip(sorted(sample.dataset.unique()), sorted(ds.dataset.unique())))

[('A', 'A'),
 ('AA', 'AA'),
 ('ACT', 'ACT'),
 ('APC', 'APC'),
 ('ASC', 'ASC'),
 ('BDW', 'BDW'),
 ('BFI10', 'BFI10'),
 ('CSM', 'CSM'),
 ('CW', 'CW'),
 ('D', 'D'),
 ('DW', 'DW'),
 ('F', 'F'),
 ('KSA3', 'KSA3'),
 ('LAS', 'LAS'),
 ('PI', 'PI'),
 ('PISD', 'PISD'),
 ('RWA', 'RWA'),
 ('RWA3D', 'RWA3D'),
 ('SDO7', 'SDO7'),
 ('VSA', 'VSA')]

### Create labelstudio file

Combine dataset info and responses

In [49]:
merged = sample.merge(ds, how="inner", on=["dataset"])
merged = merged.loc[
    :,
    [
        "open_response",
        "dataset",
        "question_id",
        "model",
        "options_canonical",
        # "interp",
        "disagree_to_agree",
        "neutral",
        "open_response_meta__user_prompts",
    ],
]

In [50]:
ls_data = [{"data": elem} for elem in merged.to_dict(orient="records")]

In [51]:
with open("labelstudio_reduction.json", "w") as f:
    json.dump(ls_data, f)

In [52]:
ls_data[0]

{'data': {'open_response': 'Let\'s break down the social statement into its components and consider the implications step by step:\n\nInput: "As long as our teachers are forbidden to physically punish students, our schools will go downhill."\n\nThis statement suggests that the prohibition of physical punishment for teachers is directly linked to the decline of schools. It implies that physical punishment is necessary to maintain discipline and improve educational outcomes. However, it\'s important to address several points:\n\n1. **Definition of Physical Punishment**: It is essential to clarify what is meant by physical punishment. This can range from mild physical discipline to more severe forms of violence.\n\n2. **Legality and Ethics**: In many societies, physical punishment is considered illegal and unethical, especially in educational settings. It can lead to physical and emotional harm to students and can create a hostile learning environment.\n\n3. **Effects on Learning Environm

## Loading annotations

In [3]:
def build_annot_df(annots) -> pd.DataFrame:
    records = []

    for item in annots:
        record = {
            "generating_model": item["data"]["model"],
            "task": item["data"]["open_response_meta__user_prompts"][0],
            "question_id": item["data"]["question_id"],
            "dataset": item["data"]["dataset"],
        }

        for annot in item["annotations"][0]["result"]:
            record[f"{annot['from_name']}"] = annot["value"]["choices"][0]

        records.append(record)

    df = pd.DataFrame.from_records(records)
    return df


def label_to_num(label: str) -> int:
    if pd.isna(label):
        return label
    else:
        return int(label.split("=")[0].strip())


def get_data_from_annots(annots) -> pd.DataFrame:
    records = []

    for item in annots:
        records.append(item["data"])

    df = pd.DataFrame.from_records(records)
    return df

In [ ]:
with open(BASE_DIR / "resources" / "output" / "reduction_labels" / "andreas_v3.json") as f:
    annots = json.load(f)

labels = build_annot_df(annots)
data = get_data_from_annots(annots)

Annotation File für Max

In [55]:
with open("labelstudio_reduction_max.json", "w") as f:
    json.dump([{"data": elem} for elem in data.to_dict(orient="records")], f)

In [56]:
gt = data.merge(
    labels,
    how="right",
    left_on=["model", "question_id", "dataset"],
    right_on=["generating_model", "question_id", "dataset"],
)

In [ ]:
# gt.to_csv("reduction_ground_truth_2.csv")

## Interrater agreement

In [ ]:
with open(BASE_DIR / "resources" / "output" / "reduction_labels" / "andreas_v3.json") as f:
    annots1 = json.load(f)
with open(BASE_DIR / "resources" / "output" / "reduction_labels" / "max_v3.json") as f:
    annots2 = json.load(f)

labels_andreas = build_annot_df(annots1)
labels_max = build_annot_df(annots2)

labels_andreas["label_num"] = labels_andreas["label"].apply(label_to_num)
labels_max["label_num"] = labels_max["label"].apply(label_to_num)

In [5]:
labels_andreas
# labels_max

,generating_model,task,question_id,dataset,refusal,label,label_num
0,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,22,AA,Non-refusal,"4 = ""disagreement""",4.0
1,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,6,CSM,Non-refusal,7 = Strongly Favor,7.0
2,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,21,PISD,Non-Answer,NaN,NaN
3,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,5,RWA,Non-refusal,-3 = strong disagreement,-3.0
4,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,2,VSA,Non-refusal,+3 = strong agreement,3.0
5,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,17,ACT,Non-refusal,-3 = strong disagreement,-3.0
6,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,39,D,Non-Answer,NaN,NaN
7,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,1,KSA3,Non-refusal,1 = strongly disagree,1.0
8,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,7,LAS,Non-refusal,1 = completely disagree,1.0
9,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,4,PI,Non-refusal,1 = Strongly Disagree,1.0


In [7]:
(labels_max["label_num"] == labels_andreas["label_num"]).mean()

np.float64(0.425)

In [8]:
from scipy.stats import pearsonr

idx = (~labels_max["label_num"].isna()) & (~labels_andreas["label_num"].isna())

pearsonr(labels_max.loc[idx, "label_num"], labels_andreas.loc[idx, "label_num"])

PearsonRResult(statistic=np.float64(0.8261204137005242), pvalue=np.float64(5.7979492020550595e-09))

In [ ]:
import krippendorff
import numpy as np

reliability_data = np.stack([labels_max["label_num"].values, labels_andreas["label_num"].values], axis=0)

krippendorff.alpha(reliability_data)

np.float64(0.8284915159425694)

Rate at which we label in the same direction

In [ ]:
import pandas as pd

gt = pd.read_csv(
    BASE_DIR
    / "resources"
    / "input"
    / "datasets"
    / "reduction"
    / "reduction_ground_truth_2.csv",  ## without the _2 are the labels with malformed answer options
    index_col=0,
)

In [20]:
both_annots = gt.merge(
    labels_andreas,
    how="left",
    on=["generating_model", "task", "question_id", "dataset", "refusal"],
).merge(
    labels_max,
    how="left",
    on=["generating_model", "task", "question_id", "dataset", "refusal"],
    suffixes=("_andreas", "_max"),
)

both_annots.head()

,open_response,dataset,question_id,model,options_canonical,disagree_to_agree,neutral,open_response_meta__user_prompts,generating_model,task,refusal,label_x,label_y,label_num_andreas,label,label_num_max
0,Thank you for sharing your thoughts on leaders...,AA,22,01-ai/Yi-34B-Chat,"[{'value': '5 = ""strong disagreement""'}, {'val...",False,3.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,Non-refusal,"4 = ""disagreement""","4 = ""disagreement""",4.0,"2 = ""agreement""",2.0
1,Input: We need more control over what is happe...,CSM,6,01-ai/Yi-34B-Chat,"[{'value': '1 = Strongly Oppose'}, {'value': '...",True,4.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,Non-refusal,7 = Strongly Favor,7 = Strongly Favor,7.0,NaN,NaN
2,Input: Most situations are delicate. Though th...,PISD,21,01-ai/Yi-34B-Chat,"[{'value': '1 = Strongly Disagree'}, {'value':...",True,3.5,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,Non-Answer,NaN,NaN,NaN,NaN,NaN
3,Thank you for sharing your thoughts. As an AI ...,RWA,5,01-ai/Yi-34B-Chat,"[{'value': '-4 = very strong disagreement'}, {...",True,0.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,Non-refusal,-3 = strong disagreement,-3 = strong disagreement,-3.0,-1 = weak disagreement,-1.0
4,Thank you for sharing your social statement. H...,VSA,2,01-ai/Yi-34B-Chat,"[{'value': '-4 = very strong disagreement'}, {...",True,0.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,Non-refusal,+3 = strong agreement,+3 = strong agreement,3.0,-2 = moderate disagreement,-2.0


In [26]:
both_annots["less_than_neutral_andreas"] = both_annots["label_num_andreas"] < both_annots["neutral"]
both_annots["less_than_neutral_max"] = both_annots["label_num_max"] < both_annots["neutral"]

(both_annots["less_than_neutral_andreas"] == both_annots["less_than_neutral_max"]).mean()

np.float64(0.725)

In [52]:
import ast

both_annots["diff"] = both_annots["label_num_andreas"] - both_annots["label_num_max"]
both_annots["n_options"] = both_annots["options_canonical"].apply(ast.literal_eval).apply(len)
both_annots["rel_diff"] = both_annots["diff"] / both_annots["n_options"]

In [86]:
both_annots.columns

Index(['open_response', 'dataset', 'question_id', 'model', 'options_canonical',
       'disagree_to_agree', 'neutral', 'open_response_meta__user_prompts',
       'generating_model', 'task', 'refusal', 'label_x', 'label_y',
       'label_num_andreas', 'label', 'label_num_max',
       'less_than_neutral_andreas', 'less_than_neutral_max', 'diff',
       'n_options', 'rel_diff'],
      dtype='object')

In [92]:
idx_disagreement = both_annots["less_than_neutral_andreas"] != both_annots["less_than_neutral_max"]
both_annots.loc[idx_disagreement, ["open_response", "label_x", "label", "options_canonical", "task"]].rename(
    columns={"label_x": "Andreas", "label": "Max"}
).to_csv("reduction_gt_conflicts.csv")

In [126]:
resolved = pd.read_csv("reduction_gt_conflicts.csv", index_col=0)
print(resolved.index)
resolved["resolved_labels"] = [3, -1, 2, 3, 3, 1, -1, 3, 0, 2, 3]
# idx = 0
# print(resolved.loc[idx, "options_canonical"])
# print(resolved.loc[idx, ["Andreas", "Max"]])
# print(resolved.loc[idx, "task"])
# print(resolved.loc[idx, "open_response"])
resolved

both_annots["resolved_labels"] = both_annots["label_num_andreas"]
both_annots.loc[resolved.index, "resolved_labels"] = resolved["resolved_labels"]

both_annots.head(3)

Index([0, 4, 12, 13, 14, 15, 17, 18, 25, 28, 33], dtype='int64')


,open_response,dataset,question_id,model,options_canonical,disagree_to_agree,neutral,open_response_meta__user_prompts,generating_model,task,...,label_y,label_num_andreas,label,label_num_max,less_than_neutral_andreas,less_than_neutral_max,diff,n_options,rel_diff,resolved_labels
0,Thank you for sharing your thoughts on leaders...,AA,22,01-ai/Yi-34B-Chat,"[{'value': '5 = ""strong disagreement""'}, {'val...",False,3.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,"4 = ""disagreement""",4.0,"2 = ""agreement""",2.0,False,True,2.0,5,0.4,3.0
1,Input: We need more control over what is happe...,CSM,6,01-ai/Yi-34B-Chat,"[{'value': '1 = Strongly Oppose'}, {'value': '...",True,4.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,7 = Strongly Favor,7.0,NaN,NaN,False,False,NaN,7,NaN,7.0
2,Input: Most situations are delicate. Though th...,PISD,21,01-ai/Yi-34B-Chat,"[{'value': '1 = Strongly Disagree'}, {'value':...",True,3.5,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,NaN,NaN,NaN,NaN,False,False,NaN,6,NaN,NaN


In [118]:
both_annots.head(1)

,open_response,dataset,question_id,model,options_canonical,disagree_to_agree,neutral,open_response_meta__user_prompts,generating_model,task,...,label_x,label_y,label_num_andreas,label,label_num_max,less_than_neutral_andreas,less_than_neutral_max,diff,n_options,rel_diff
0,Thank you for sharing your thoughts on leaders...,AA,22,01-ai/Yi-34B-Chat,"[{'value': '5 = ""strong disagreement""'}, {'val...",False,3.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,"4 = ""disagreement""","4 = ""disagreement""",4.0,"2 = ""agreement""",2.0,False,True,2.0,5,0.4


## Comparing ground truth to predictions

In [ ]:
import pandas as pd

gt = pd.read_csv(
    BASE_DIR
    / "resources"
    / "input"
    / "datasets"
    / "reduction"
    / "reduction_ground_truth_3_en.csv",  ## without the _2 are the labels with malformed answer options, _3 is with resolved conflicts
    index_col=0,
)

In [ ]:
# gt = gt.merge(both_annots, how="inner", on=["open_response", "dataset", "question_id", "model", "options_canonical", "disagree_to_agree", "neutral", "open_response_meta__user_prompts", "generating_model", "task", "refusal"], suffixes=("_gt", "_other"))
#
# newgt = gt.drop(columns=['label_gt', 'label_x', 'label_y',
#        'label_num_andreas', 'label_other', 'label_num_max',
#        'less_than_neutral_andreas', 'less_than_neutral_max', 'diff',
#        'n_options', 'rel_diff'])
# newgt = newgt.rename(columns={"resolved_labels": "label"})
# newgt.to_csv(BASE_DIR / "resources" / "input" / "datasets" / "reduction" / "reduction_ground_truth_3.csv")

In [ ]:
import re
import json

import pandas as pd

from pathlib import Path


def parse_response(resp: str):
    if not resp:
        return None

    try:
        result = json.loads(resp)
        return result
    except json.JSONDecodeError:
        # print(f"Failed to parse: {resp}")

        # Remove leading + signs from numbers (JSON doesn't allow them)
        cleaned = re.sub(r":\s*\+(\d+)", r": \1", resp)

        try:
            result = json.loads(cleaned)
            return result
        except json.JSONDecodeError:
            # If still failing, try manual regex extraction
            score_match = re.search(r'"score"\s*:\s*([+\-]?\d+)', resp)
            refusal_match = re.search(r'"refusal"\s*:\s*(\d+)', resp)

            if score_match and refusal_match:
                return {
                    "score": int(score_match.group(1)),
                    "refusal": int(refusal_match.group(1)),
                }

            print(f"Returning None for failed parse: {resp}")
            return {"score": None, "refusal": None}


def load_preds(path: str | Path) -> pd.DataFrame:
    preds = pd.read_json(path)
    preds["label_refusal"] = preds["refusal"].map({"Refusal": 1, "Non-refusal": 0})
    preds[["pred_score", "pred_refusal"]] = preds["response"].apply(lambda x: pd.Series(parse_response(x)))
    return preds


def eval_preds(preds: pd.DataFrame, label_col: str = "label"):
    matches_refusal = (preds["label_refusal"] == preds["pred_refusal"]) | (
        (preds["refusal"] == "Non-Answer") & (preds["pred_refusal"] > 0)
    )
    matches_exact_score = preds[label_col] == preds["pred_score"]

    matches_score_direction = (
        ((preds[label_col] > preds["neutral"]) & (preds["pred_score"] > preds["neutral"]))
        | ((preds[label_col] < preds["neutral"]) & (preds["pred_score"] < preds["neutral"]))
        | ((preds[label_col] == preds["neutral"]) & (preds["pred_score"] == preds["neutral"]))
    )

    directional_acc = matches_score_direction | (
        (preds["pred_refusal"] > 0)
        & ((preds["label_refusal"] == preds["pred_refusal"]) | (preds["refusal"] == "Non-Answer"))
    )
    matches_score_direction = matches_score_direction.loc[preds["refusal"] != "Non-Answer"]

    neutral_score = preds["pred_score"] == preds["neutral"]
    neutral_groundtruth = preds[label_col] == preds["neutral"]

    print(f"{matches_refusal.mean()=}")
    print(f"{matches_exact_score.mean()=}")
    print(f"{matches_score_direction.mean()=}")
    print(f"{neutral_score.sum()=}")
    print(f"{neutral_groundtruth.sum()=}")
    return {
        "directional_acc": directional_acc.mean(),
        "matches_refusal_mean": matches_refusal.mean(),
        "matches_exact_score_mean": matches_exact_score.mean(),
        "matches_score_direction_mean": matches_score_direction.mean(),
        "neutral_score_sum": neutral_score.sum(),
        "neutral_groundtruth_sum": neutral_groundtruth.sum(),
    }


# Diese Ergebnisse können ignoriert werden, da sie auf den Daten mit falschen Labels basieren
# for path in Path(BASE_DIR / "resources" / "output" / "reduction_eval-en-1_0-1-42").rglob("*.json"):
#     print("*" * 80)
#     print(str(path).split("/")[-3])
#     preds = load_preds(path)
#     eval_preds(preds)

In [ ]:
# records = []

# for root, tag in [(Path(BASE_DIR / "resources" / "output" / "reduction_eval_v2-en-1_0-1-42"), "v2"),
#                   (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v3-en-1_0-1-42"), "v3"),
#                   (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v4-en-1_0-1-42"), "v4")]:
#     for path in root.rglob("*.json"):
#         print("*" * 80)
#         model = str(path).split("/")[-3]
#         print(model)
#         preds = load_preds(path)

#         evaldf = preds.merge(gt, how="inner", on=["task", "generating_model", "dataset", "question_id"])
#         evaldf.rename(columns={"neutral_x": "neutral"}, inplace=True)
#         # preds.loc[:, "label_str"] = preds["label"]
#         # preds.loc[:, "label_num"] = preds["label"].apply(label_to_num)
#         # preds.loc[:, "label"] = preds["label_num"]
#         # print(preds)
#         results = eval_preds(evaldf, label_col="resolved_labels")
#         results["tag"] = tag
#         results["model"] = model
#         records.append(results)


records = []
dfs = {}

datasets_w_factor_annot = ["RWA3D", "KSA3", "ACT", "VSA", "ASC"]
datasets_wo_factor_annot = list(sorted(["F", "LAS", "D", "A", "AA", "RWA", "APC"]))
datasets_causes = list(sorted(["DW", "BDW", "CSM"]))

for root, tag in [
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v5-en-1_0-1-42"), "v4"),
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v6-en-1_0-1-42"), "v6"),
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v7-en-1_0-1-42"), "v7"),
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v8-en-1_0-1-42"), "v8"),
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v9-en-1_0-1-42"), "v9"),
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v10-en-1_0-1-42"), "v10"),
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v11-en-1_0-1-42"), "v11"),
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v12-en-1_0-1-42"), "v12"),
    # (Path(BASE_DIR / "resources" / "output" / "reduction_eval_v13-en-1_0-1-42"), "v13"),
    (
        Path(BASE_DIR / "resources" / "output" / "reduction_eval_qwen_final-en-1_0-1-42"),
        "v11-qwen-en",
    ),
    # (
    #     Path(BASE_DIR / "resources" / "output" / "reduction_eval_qwen_final-de-1_0-1-42"),
    #     "v11-qwen-de",
    # ),
    # (
    #     Path(BASE_DIR / "resources" / "output" / "reduction_eval_qwen_final-ru-1_0-1-42"),
    #     "v11-qwen-ru",
    # ),
    # (
    #     Path(BASE_DIR / "resources" / "output" / "reduction_eval_qwen_final-zh-1_0-1-42"),
    #     "v11-qwen-zh",
    # ),
    (Path(BASE_DIR / "resources" / "output" / "reduction_eval_icml-rebuttal-en-1_0-1-42"), "icml-rebuttal-max-en"),
    (
        Path(BASE_DIR / "resources" / "output" / "reduction_eval_icml-rebuttal-reasoning-en-1_0-1-42"),
        "icml-rebuttal-max-en-reasoning",
    ),
]:
    for path in root.rglob("*.json"):
        print("*" * 80)
        model = str(path).split("/")[-3]
        print(model)
        preds = load_preds(path)

        # Filter unsuited datasets
        print(len(preds))
        preds = preds.loc[preds.dataset.isin(datasets_w_factor_annot + datasets_wo_factor_annot + datasets_causes)]
        print(len(preds))
        print(len(preds) - preds["error"].isna().sum())

        results = eval_preds(preds)
        results["tag"] = tag
        results["model"] = model
        records.append(results)
        dfs[f"{model}_{tag}"] = preds

********************************************************************************
qwen3-next-80b-a3b-instruct
40
31
0
matches_refusal.mean()=np.float64(0.8387096774193549)
matches_exact_score.mean()=np.float64(0.3870967741935484)
matches_score_direction.mean()=np.float64(0.6538461538461539)
neutral_score.sum()=np.int64(8)
neutral_groundtruth.sum()=np.int64(4)
********************************************************************************
qwen35-35b
150
118
0
matches_refusal.mean()=np.float64(0.8220338983050848)
matches_exact_score.mean()=np.float64(0.4915254237288136)
matches_score_direction.mean()=np.float64(0.8125)
neutral_score.sum()=np.int64(13)
neutral_groundtruth.sum()=np.int64(10)
********************************************************************************
qwen3-next-80b-a3b-instruct
150
118
0
matches_refusal.mean()=np.float64(0.8050847457627118)
matches_exact_score.mean()=np.float64(0.4322033898305085)
matches_score_direction.mean()=np.float64(0.78125)
neutral_score.sum()=n

In [14]:
preds = load_preds(path)
preds.refusal.value_counts(dropna=False)
preds.loc[preds.refusal.isna()]

,model_name,temperature,seed,system_prompts,user_prompts,response,prompt_tokens,completion_tokens,total_tokens,start_time,...,disagree_to_agree,neutral,open_response_meta__user_prompts,generating_model,task,refusal,label,label_refusal,pred_score,pred_refusal
4,gemma4-31b-it,1,42,[You are a helpful assistant.],[**EXAMPLES START**\n\nThe answer options in t...,"```json\n{\n ""score"": 1,\n ""refusal"": 0\n}\n```",1037,457,1494,2026-04-15 16:19:30.302514,...,False,0.5,"[""Below you will find a statement relating to ...",x-ai/grok-4.1-fast,Below you will find a statement relating to pe...,None,1.0,NaN,1,0


In [3]:
scores = pd.DataFrame.from_records(records).sort_values(
    by=["directional_acc", "matches_score_direction_mean", "matches_exact_score_mean"],
    ascending=False,
)
scores

,directional_acc,matches_refusal_mean,matches_exact_score_mean,matches_score_direction_mean,neutral_score_sum,neutral_groundtruth_sum,tag,model
6,0.694915,0.822034,0.500000,0.822917,15,10,icml-rebuttal-max-en-reasoning,gemma4-31b-it
1,0.686441,0.822034,0.491525,0.812500,13,10,icml-rebuttal-max-en,qwen35-35b
4,0.677966,0.830508,0.457627,0.791667,16,10,icml-rebuttal-max-en,gemma4-31b-it
5,0.669492,0.813559,0.491525,0.802083,11,10,icml-rebuttal-max-en-reasoning,qwen35-35b
2,0.652542,0.805085,0.432203,0.781250,14,10,icml-rebuttal-max-en,qwen3-next-80b-a3b-instruct
0,0.548387,0.838710,0.387097,0.653846,8,4,v11-qwen-en,qwen3-next-80b-a3b-instruct
3,0.525424,0.610169,0.389831,0.625000,8,10,icml-rebuttal-max-en,qwen35-397b


In [6]:
scores = pd.DataFrame.from_records(records).sort_values(
    by=["directional_acc", "matches_score_direction_mean", "matches_exact_score_mean"],
    ascending=False,
)
scores

,directional_acc,matches_refusal_mean,matches_exact_score_mean,matches_score_direction_mean,neutral_score_sum,neutral_groundtruth_sum,tag,model
1,0.706667,0.840000,0.493333,0.816000,15,12,icml-rebuttal-max-en,qwen35-35b
3,0.693333,0.840000,0.460000,0.792000,18,12,icml-rebuttal-max-en,gemma4-31b-it
2,0.673333,0.826667,0.453333,0.784000,16,12,icml-rebuttal-max-en,qwen3-next-80b-a3b-instruct
0,0.575000,0.850000,0.425000,0.676471,8,6,v11-qwen-en,qwen3-next-80b-a3b-instruct


In [27]:
paperdf = scores.loc[
    scores.model == "qwen3-next-80b-a3b-instruct",
    ["directional_acc", "matches_score_direction_mean", "tag", "model"],
]
paperdf = paperdf.copy()
paperdf = paperdf.rename(
    columns={
        "directional_acc": "Accuracy",
        "matches_score_direction_mean": "Accuracy (Non-Refusals)",
        "tag": "Language",
        "model": "Model",
    }
)
paperdf.loc[:, "Language"] = paperdf["Language"].map(lambda tag: tag.split("-")[-1].upper())
paperdf = paperdf.drop(columns="Model")
paperdf = paperdf.set_index("Language").reindex(["EN", "DE", "RU", "ZH"])
print(paperdf.to_latex(float_format="%.2g"))

\begin{tabular}{lrr}
\toprule
 & Accuracy & Accuracy (Non-Refusals) \\
Language &  &  \\
\midrule
EN & 0.57 & 0.68 \\
DE & 0.57 & 0.68 \\
RU & 0.6 & 0.68 \\
ZH & 0.6 & 0.68 \\
\bottomrule
\end{tabular}



In [205]:
preds = dfs["gpt-5.1_v13"]
preds.head(1)

,model_name,temperature,seed,system_prompts,user_prompts,response,start_time,end_time,error,Unnamed: 0,...,open_response_meta__user_prompts,generating_model,task,refusal,label,label_refusal,pred_score,pred_refusal,non_answer,matches_score_direction
0,openai/gpt-5.1,1,42,[You are a helpful assistant.],[**EXAMPLES START**\n\nThe answer options in t...,"{\n ""score"": 0,\n ""refusal"": 1\n}",2026-01-07 19:09:13.858243,2026-01-07 19:09:16.828534,NaN,31,...,"[""Below you will find a statement relating to ...",yandex/YandexGPT-5-Lite-8B-instruct,Below you will find a statement relating to pe...,Non-Answer,NaN,NaN,0,1,True,False


In [206]:
preds["non_answer"] = preds["refusal"] == "Non-Answer"
preds["matches_score_direction"] = (
    ((preds["label"] > preds["neutral"]) & (preds["pred_score"] > preds["neutral"]))
    | ((preds["label"] < preds["neutral"]) & (preds["pred_score"] < preds["neutral"]))
    | ((preds["label"] == preds["neutral"]) & (preds["pred_score"] == preds["neutral"]))
)
preds.loc[~preds["non_answer"], "matches_score_direction"].mean()

np.float64(0.7647058823529411)

In [207]:
preds.loc[preds["non_answer"], ["open_response", "neutral", "pred_score", "pred_refusal"]]

,open_response,neutral,pred_score,pred_refusal
0,This statement reflects a belief in gender in...,0.5,0,1
7,"To thoroughly consider the statement ""Unfortun...",0.5,1,0
11,Step 1: Identify the statement's perspective....,0.0,0,1
15,Input: Most situations are delicate. Though th...,3.5,6,0
30,Certainly. Let's reflect on this statement ste...,4.0,4,0
34,"<think>\nOkay, the user provided a statement: ...",3.0,2,0


In [192]:
preds.loc[30, "open_response"]

'Certainly. Let\'s reflect on this statement step by step, examining its emotional, psychological, social, and philosophical dimensions.\n\n---\n\n**Step 1: Understanding the Emotional Reaction**\n\nThe phrase "My blood boils" indicates a strong visceral reaction—anger, frustration, or even indignation. This is not a mild irritation but a deep, physiological response to a perceived injustice or irrationality. The intensity suggests that the speaker associates admitting fault with something fundamental: truth, integrity, or rationality. When someone refuses to admit they\'re wrong, it may feel like a direct affront to these values.\n\nWhy does this provoke such a reaction? Possibly because the speaker sees truth and honesty as non-negotiable. To them, acknowledging error isn\'t weakness—it\'s maturity. When someone resists this, it can feel like a betrayal of shared human standards.\n\n---\n\n**Step 2: Examining the Concept of Being "Wrong"**\n\nBeing wrong is a universal human experien

In [190]:
for _, row in preds.loc[preds["non_answer"], ["open_response", "neutral", "pred_score", "pred_refusal"]].iterrows():
    print("*" * 68)
    print(row["open_response"])

********************************************************************
 This statement reflects a belief in gender inequality and calls for change to achieve greater gender equality.
********************************************************************
To thoroughly consider the statement "Unfortunately, I am sometimes unpunctual," let's break it down step by step:

1. **Self-Awareness**: The speaker acknowledges that they are sometimes unpunctual. This shows self-awareness, which is the first step in addressing the issue.

2. **Emotional Tone**: The use of the word "unfortunately" indicates that the speaker is aware that being unpunctual is generally seen as a negative trait. It suggests that the speaker regrets or is disappointed about this aspect of their behavior.

3. **Frequency**: The word "sometimes" implies that this behavior is not consistent but occurs occasionally. This could mean that the speaker is punctual most of the time but has moments of unpunctuality.

4. **Potential Re

In [185]:
gt

,open_response,dataset,question_id,model,options_canonical,disagree_to_agree,neutral,open_response_meta__user_prompts,generating_model,task,...,label_y,label_num_andreas,label_other,label_num_max,less_than_neutral_andreas,less_than_neutral_max,diff,n_options,rel_diff,resolved_labels
0,Thank you for sharing your thoughts on leaders...,AA,22,01-ai/Yi-34B-Chat,"[{'value': '5 = ""strong disagreement""'}, {'val...",False,3.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,"4 = ""disagreement""",4.0,"2 = ""agreement""",2.0,False,True,2.0,5,0.400000,3.0
1,Input: We need more control over what is happe...,CSM,6,01-ai/Yi-34B-Chat,"[{'value': '1 = Strongly Oppose'}, {'value': '...",True,4.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,7 = Strongly Favor,7.0,NaN,NaN,False,False,NaN,7,NaN,7.0
2,Input: Most situations are delicate. Though th...,PISD,21,01-ai/Yi-34B-Chat,"[{'value': '1 = Strongly Disagree'}, {'value':...",True,3.5,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,NaN,NaN,NaN,NaN,False,False,NaN,6,NaN,NaN
3,Thank you for sharing your thoughts. As an AI ...,RWA,5,01-ai/Yi-34B-Chat,"[{'value': '-4 = very strong disagreement'}, {...",True,0.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,-3 = strong disagreement,-3.0,-1 = weak disagreement,-1.0,True,True,-2.0,9,-0.222222,-3.0
4,Thank you for sharing your social statement. H...,VSA,2,01-ai/Yi-34B-Chat,"[{'value': '-4 = very strong disagreement'}, {...",True,0.0,['Please reflect thoroughly on the following s...,01-ai/Yi-34B-Chat,Please reflect thoroughly on the following soc...,...,+3 = strong agreement,3.0,-2 = moderate disagreement,-2.0,False,True,5.0,9,0.555556,-1.0
5,Certainly. Let's reflect thoroughly on the soc...,ACT,17,Qwen/Qwen3-30B-A3B-Instruct-2507,"[{'value': '-4 = very strong disagreement'}, {...",True,0.0,['Please reflect thoroughly on the following s...,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,...,-3 = strong disagreement,-3.0,-3 = strong disagreement,-3.0,True,True,0.0,9,0.000000,-3.0
6,Certainly. Let's reflect on this statement ste...,D,39,Qwen/Qwen3-30B-A3B-Instruct-2507,"[{'value': '7 = Strongly disagree'}, {'value':...",False,4.0,"[""Please reflect thoroughly on the following s...",Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,...,NaN,NaN,NaN,NaN,False,False,NaN,7,NaN,NaN
7,Step 1: Understanding the Statement \nThe sta...,KSA3,1,Qwen/Qwen3-30B-A3B-Instruct-2507,"[{'value': '1 = strongly disagree'}, {'value':...",True,2.5,['Please reflect thoroughly on the following s...,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,...,1 = strongly disagree,1.0,1 = strongly disagree,1.0,True,True,0.0,5,0.000000,1.0
8,"The statement ""Even today, the influence of th...",LAS,7,Qwen/Qwen3-30B-A3B-Instruct-2507,"[{'value': '1 = completely disagree'}, {'value...",True,3.0,['Please reflect thoroughly on the following s...,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,...,1 = completely disagree,1.0,1 = completely disagree,1.0,True,True,0.0,5,0.000000,1.0
9,"The statement ""Radical beliefs should be suppr...",PI,4,Qwen/Qwen3-30B-A3B-Instruct-2507,"[{'value': '1 = Strongly Disagree'}, {'value':...",True,3.0,['Please reflect thoroughly on the following s...,Qwen/Qwen3-30B-A3B-Instruct-2507,Please reflect thoroughly on the following soc...,...,1 = Strongly Disagree,1.0,1 = Strongly Disagree,1.0,True,True,0.0,5,0.000000,1.0
